# ANALYSE-09: A single tile of Z^3 that tiles without full periodicity

**Series**: SymbolicAI / Lean — Digestions of deep results, companion to the capstone [Lean-20 — Digestions Tao](../Lean-20-Capstone-Digestions-Tao-Python.ipynb)
**Source**: family 155 of the openai/math corpus ("A counterexample to periodic tiling in dimension three", published Lean skeleton) — reconnaissance repository `G:\Mon Drive\MyIA\IA\Distillations\2026-10-openai-math\`
**Source lake**: `sudoku_lean` (the repository's exact-cover / discrete tiling semantics, `MyIA.AI.Notebooks/Sudoku/sudoku_lean/`)
**Kernel**: `python3` — this notebook illustrates the definitions and measures toy cases; the formal statement remains the corpus's

## Navigation

| Previous notebook | Next notebook |
|---|---|
| [ANALYSE-04 - PFR Primitives](ANALYSE-04-PFR-Primitives-Python.ipynb) | — (this notebook is the current head of the series) |

***

## Why this notebook

The periodic tiling conjecture says, roughly: *if a shape tiles space, it tiles it periodically*. It is true in dimension 1, true for a single tile in dimension 2 (on Z^2), and **false from dimension 3 onward**: family 155 of the openai/math corpus exhibits a **single tile** — a finite set T ⊂ Z^3 — that tiles Z^3 by translations but for which **no tiling is fully periodic**.

This is the counterexample in the **smallest possible dimension** for a single tile. The unit-cube thickening carries the same counterexample over to R^3, even allowing arbitrary real translation vectors.

> **Epistemic status**: openai/math is a third-party source not verified by us. The statement above is the one declared by the corpus (published Lean skeleton for family 155); this notebook **presents** it and **illustrates** it on measured toy cases — it does not re-prove the theorem.

## What the notebook establishes (plan)

1. **§1** — the exact definitions: tile, tiling by translations, period, *full periodicity* (rank 3) — and why "aperiodic" is subtler than "without any period".
2. **§2** — the exact-cover connection: tiling a finite window is solving an exact cover — the semantics that the `sudoku_lean` lake formalizes.
3. **§3** — the engine (bounded backtracking + period-rank measurement) and four measured experiments: a periodic control, a tile that tiles **no** box (and why that is a theorem), a tile with a **proven** minimal period of exactly 4, and a trap: a tile that **looks** aperiodic on a small window.
4. **§4** — honest reading: what finite windows can and cannot establish, with the §3 measurements as evidence.


## 1. Definitions — tile, tiling, full periodicity

**Tile.** A *tile* is a nonempty **finite** set T ⊂ Z^3. Its elements are cells; one thinks of a connected polycube, but the definition does not require connectivity.

**Tiling by translations.** The tile T *tiles* Z^3 if there exists a set of translations S ⊂ Z^3 such that the copies T + s (s ∈ S) cover Z^3 **exactly**: every cell of Z^3 belongs to one and only one copy. This is an **exact cover** of Z^3 by translated copies of T — no hole, no overlap.

**Period, periodic tiling.** A vector v ∈ Z^3 \ {0} is a *period* of a tiling if its set of translations is invariant under v: translating the whole tiling by v leaves it unchanged. The **period group** of a tiling is a subgroup of Z^3; it has a **rank** r ∈ {0, 1, 2, 3} (dimension of the Z-module).

**Full periodicity.** A tiling is *fully periodic* when r = 3: three independent periods. This is the notion that theorem 155 strikes down: the counterexample tile tiles Z^3, but **none** of its tilings reaches full rank.

**Why this nuance matters.** "Aperiodic" does not mean "zero periods": a tiling may admit one or two directions of periods (rank 1 or 2) and still be a counterexample — what is forbidden is full rank. A rank-2 tiling, for instance, repeats in two directions but never in the third: enough to refute the conjecture, not enough to be "structureless".


## 2. The exact-cover connection — what `sudoku_lean` formalizes

On a **finite window** W (an N₁×N₂×N₃ block of Z^3), tiling W by copies of T is an exact cover problem: every cell of W must be covered **exactly once**, every tile placement is a "choice", and the choices must be pairwise non-conflicting.

This is exactly the semantics of the repository's `sudoku_lean` lake: `ExactCover.IsExactCover` (`MyIA.AI.Notebooks/Sudoku/sudoku_lean/Sudoku/ExactCover.lean`) defines the exact cover of a set of *scopes* by a selection, and Sudoku is the canonical instance there. Sudoku is a tiling: every cell receives exactly one digit, every constraint is covered exactly once. Our tiling window is another instance of the same semantics — cells × placements instead of cells × values.

> **Declared pedagogical copy** (organ-first rule of the repository): the bounded solver in this notebook is a **declared** local reimplementation — plain, readable Python backtracking — of the exact-cover semantics. The native organ (`sudoku_lean`) is a Lean proof lake, not a consumable Python library; the real invocation is unavailable on the Python side, and the goal here is to show the mechanics, not to prove them. Answers to the 5 questions: the Sudoku series owns the semantics (1); its module is not invocable from Python (2); exporting it would require a port with no pedagogical value here (3); the organ's negative witness is the formal proof `mem_toSelection_iff`, out of reach for a notebook (4); the independent verification is the multi-window replay of §3, whose measured ranks confront the hand-proven constraints (5).


## 3. The engine and the four experiments

The engine has two organs, both deterministic:

- `place_tiles(window, tile)` — bounded exact-cover **backtracking**: the first free cell (lexicographic order) forces the choice of the anchor covering it, descend, backtrack. The node counter measures the combinatorial cost.
- `period_rank(placement, window)` — for the tiling obtained, tests all period vectors v in the box |vᵢ| ≤ ⌊Nᵢ/2⌋ (the **half-window**) and returns the Z-rank of the group they generate (exact Gaussian elimination over the rationals).

The half-window is the reading key to all of §3: **any period longer than the half-window is invisible**. This is a measurement ceiling, not a property of the tile — experiments A and B2 each show it in their own way.

Four experiments:

| # | Tile | What is measured | Hand-provable constraint |
|---|---|---|---|
| A | 1×1×2 domino | the periodic control: rank 3 expected | obvious tiling, periods 1 or 2 |
| B0 | L×{0} prism | the tiles **no** aligned box | theorem: forced corner, induction, dead cell (1,1) |
| B1 | gapped domino {(0,0,0),(2,0,0)} | rank 2 then rank 3 as the window grows | minimal x-period **exactly 4** in every tiling |
| B2 | doubled cube {0,2}³ | rank 0 then rank 3 | **unique** tiling of Z^3, periods exactly 4Z³ |


In [1]:
"""Moteur de calcul pour le notebook ANALYSE-09-Tuilage-Aperiodique (famille 155).

Famille 155 : « contre-exemple au tuilage périodique en dimension trois » —
il existe une tuile translative finie T sous-ensemble de Z^3 qui tuile Z^3
mais n'admet AUCUN tuilage pleinement périodique (groupe de périodes de rang
3) ; des périodes partielles de rang < 3 peuvent exister. La tuile du théorème
est énorme : ce moteur fournit donc des mesures BORNÉES et HONNÊTES sur de
petites fenêtres.

Deux organes :
  * place_tiles  : exact-cover par backtracking borné (translations seules,
                   aucune rotation), déterministe, avec compteur de noeuds ;
  * period_rank  : rang sur Z du groupe des vecteurs de période du tuilage
                   RESTREINT à la fenêtre, testés dans |v_i| <= floor(N_i/2).

Limite pédagogique assumée (aussi dans tiling_measurements.json, champ
« limitation ») : une petite fenêtre ne peut jamais PROUVER l'apériodicité,
elle peut seulement la suggérer — une fenêtre plus grande peut révéler des
périodes jusqu'alors invisibles. La preuve du théorème vit dans la prose du
notebook, pas dans ces mesures.
"""

from __future__ import annotations

from fractions import Fraction
from itertools import product
from typing import Iterable, Optional

import numpy as np

Cell = tuple[int, int, int]
Placement = dict[Cell, Cell]  # cellule -> ancre de la copie qui la couvre



class _NodeBudgetExceeded(RuntimeError):
    """Budget de noeuds épuisé : instance abandonnée (ni échec ni succès)."""


def place_tiles(
    window_shape: tuple[int, int, int],
    tile_cells: Iterable[Cell],
    forbidden: Optional[set[Cell]] = None,
    *,
    max_nodes: int = 2_000_000,
) -> tuple[Optional[Placement], int]:
    """Tente de tuiler exactement la fenêtre par copies translatées de la tuile.

    Parameters
    ----------
    window_shape : (N1, N2, N3) — boîte [0,N1) x [0,N2) x [0,N3).
    tile_cells   : décalages (x,y,z) de la tuile, relatifs à son ancre.
    forbidden    : cellules à laisser vides (trous), optionnel.
    max_nodes    : garde-fou sur le nombre de noeuds de backtracking.

    Retour
    ------
    (placement, nodes) où placement vaut None si aucun tuilage n'est trouvé
    (ou si le budget de noeuds est épuisé). placement associe à chaque cellule
    couverte l'ancre de la copie qui la couvre — support direct du test de
    période de period_rank.

    Déterminisme : la première cellule libre est prise en ordre lexicographique
    (ordre C : x lent, z rapide) et les ancre candidates sont essayées par
    ordre croissant. Aucun aléatoire.
    """
    n1, n2, n3 = (int(v) for v in window_shape)
    if min(n1, n2, n3) < 1:
        raise ValueError("window_shape doit valoir >= 1 sur chaque axe")
    cells: list[Cell] = sorted({(int(x), int(y), int(z)) for x, y, z in tile_cells})
    if not cells:
        raise ValueError("tile_cells est vide")

    occ = np.zeros((n1, n2, n3), dtype=np.uint8)  # 0 = libre, 1 = couvert, 2 = trou
    if forbidden:
        for (x, y, z) in forbidden:
            occ[int(x), int(y), int(z)] = 2

    n_free = int((occ == 0).sum())
    if n_free % len(cells) != 0:
        return None, 1  # volume incompatible avec |T| : échec immédiat, décidé.

    anchors: list[Cell] = []
    nodes = 0

    def candidates(c: Cell) -> list[Cell]:
        """Ancres valides couvrant la cellule c, triées (déterminisme)."""
        out: list[Cell] = []
        for t in cells:
            a: Cell = (c[0] - t[0], c[1] - t[1], c[2] - t[2])
            if a in out:
                continue
            ok = True
            for u in cells:
                p = (a[0] + u[0], a[1] + u[1], a[2] + u[2])
                if not (0 <= p[0] < n1 and 0 <= p[1] < n2 and 0 <= p[2] < n3):
                    ok = False
                    break
                if occ[p] != 0:
                    ok = False
                    break
            if ok:
                out.append(a)
        return sorted(out)

    def solve() -> bool:
        nonlocal nodes
        nodes += 1
        if nodes > max_nodes:
            raise _NodeBudgetExceeded
        flat = occ.reshape(-1)
        k = int(np.argmin(flat))  # première cellule libre en ordre C
        if int(flat[k]) != 0:
            return True  # plus aucune cellule libre : succès
        c3 = np.unravel_index(k, (n1, n2, n3))
        c: Cell = (int(c3[0]), int(c3[1]), int(c3[2]))
        for a in candidates(c):
            placed = [(a[0] + u[0], a[1] + u[1], a[2] + u[2]) for u in cells]
            for p in placed:
                occ[p] = 1
            anchors.append(a)
            if solve():
                return True
            anchors.pop()
            for p in placed:
                occ[p] = 0
        return False

    try:
        solved = solve()
    except _NodeBudgetExceeded:
        return None, nodes
    if not solved:
        return None, nodes

    placement: Placement = {}
    for a in anchors:
        for u in cells:
            placement[(a[0] + u[0], a[1] + u[1], a[2] + u[2])] = a
    return placement, nodes


def period_rank(
    placement: Placement, window_shape: tuple[int, int, int]
) -> tuple[int, list[Cell]]:
    """Rang sur Z du groupe des périodes du tuilage, vues DANS la fenêtre.

    v est une période si, pour toute cellule c telle que c+v soit aussi dans
    la fenêtre, placement[c+v] == placement[c] + v (la copie qui couvre c,
    translatée de v, couvre c+v). Boîte de test bornée : |v_i| <= floor(N_i/2)
    (borne heuristique du sujet) ; les vecteurs v et -v sont identifiés et
    chaque période est mise sous forme canonique (premier coefficient non nul
    positif).

    Retour : (rang, generateurs) — sous-ensemble libre maximal pris dans
    l'ordre croissant (élimination gaussienne exacte sur Q via Fraction).
    """
    n1, n2, n3 = (int(v) for v in window_shape)
    anchor = np.zeros((n1, n2, n3, 3), dtype=np.int64)
    mask = np.zeros((n1, n2, n3), dtype=bool)  # cellules présentes dans le placement
    for c, a in placement.items():
        anchor[c] = a
        mask[c] = True

    dims = (n1, n2, n3)
    periods: list[Cell] = []
    ranges = product(*(range(-(n // 2), n // 2 + 1) for n in dims))
    for v in ranges:
        if v == (0, 0, 0):
            continue
        if any(n - abs(vi) < 1 for n, vi in zip(dims, v)):
            continue  # chevauchement vide : « période » vide de sens, on ignore
        src: list[slice] = []
        dst: list[slice] = []
        for n_i, v_i in zip(dims, v):
            if v_i >= 0:
                src.append(slice(0, n_i - v_i))
                dst.append(slice(v_i, n_i))
            else:
                src.append(slice(-v_i, n_i))
                dst.append(slice(0, n_i + v_i))
        shifted = anchor[tuple(dst)]
        base = anchor[tuple(src)] + np.array(v, dtype=np.int64)
        both = mask[tuple(dst)] & mask[tuple(src)]
        if not bool(np.any(both)):
            continue  # chevauchement réduit aux seuls trous : période vide de sens
        eq = np.all(shifted == base, axis=-1)  # égalité des ancres, axe vec inclus
        if bool(np.all(eq | ~both)):
            w = tuple(int(x) for x in v)
            for x in w:  # forme canonique : premier coefficient non nul positif
                if x < 0:
                    w = tuple(-y for y in w)
                    break
                if x > 0:
                    break
            if w not in periods:
                periods.append(w)

    # Préférence pour les générateurs courts (norme L1, puis ordre) : pure
    # lisibilité pédagogique, sans effet sur le rang ni le déterminisme.
    periods.sort(key=lambda w: (sum(abs(x) for x in w), w))
    gens = _independent_basis(periods)
    return len(gens), gens


def _independent_basis(vectors: list[Cell]) -> list[Cell]:
    """Sous-ensemble libre maximal, par élimination gaussienne exacte sur Q."""
    rows: list[list[Fraction]] = []
    pivots: list[int] = []
    kept: list[Cell] = []
    for v in vectors:
        row = [Fraction(x) for x in v]
        for piv, r in zip(pivots, rows):
            f = row[piv]
            if f:
                row = [a - f * b for a, b in zip(row, r)]
        for j, val in enumerate(row):
            if val:
                row = [a / val for a in row]
                rows.append(row)
                pivots.append(j)
                kept.append(v)
                break
    return kept


In [2]:
def mesurer(nom: str, cells: set[Cell], win: tuple[int, int, int]) -> dict[str, object]:
    """Resout une fenetre, mesure le rang des periodes, imprime et retourne."""
    placement, nodes = place_tiles(win, cells)
    entry: dict[str, object] = {
        "tile": nom, "cells": sorted(cells), "window": list(win),
        "nodes": nodes, "solved": placement is not None,
    }
    if placement is None:
        print(f" {nom} | fenetre {win} | nodes {nodes} | NON RESOLU")
        return entry
    rang, gens = period_rank(placement, win)
    entry["period_rank"] = rang
    entry["generators"] = [list(g) for g in gens]
    print(f" {nom} | fenetre {win} | nodes {nodes} | rang {rang} | generateurs {gens}")
    return entry


### 3.1 Experiment A — the domino, periodic control

Two windows: a comfortable 4×4×4 cube, and a deliberately **thin** (6,4,2) window — the domino's period (0,0,2) exceeds the z half-window there (= 1).


In [3]:
# A -- domino 1x1x2 : controle periodique.
domino: set[Cell] = {(0, 0, 0), (0, 0, 1)}
a_confort = mesurer("domino", domino, (4, 4, 4))
a_mince = mesurer("domino", domino, (6, 4, 2))


 domino | fenetre (4, 4, 4) | nodes 33 | rang 3 | generateurs [(0, 1, 0), (1, 0, 0), (0, 0, 2)]
 domino | fenetre (6, 4, 2) | nodes 25 | rang 2 | generateurs [(0, 1, 0), (1, 0, 0)]


### Reading the result

On the comfortable window (4,4,4), the observed period group has **rank 3** — three independent generators including (0,0,2): the domino is fully periodic, and the measurement sees it. On the thin window (6,4,2), the measured rank drops to **2** although the tile has not changed and remains periodic: the period (0,0,2) exceeds the z half-window (⌊2/2⌋ = 1) and becomes **invisible**. The periodic control thus fails to show itself periodic when the window is too small — this is the notebook's first measurement lesson: *the measured rank is a lower bound on the true rank, capped by the half-window*.


### 3.2 Experiment B0 — the L prism: a search failure that is a theorem

The L triomino of the xy plane ({(0,0,0), (1,0,0), (0,1,0)}), prism-ed with thickness 1 in z. We run it on three growing windows — it solves **none**, in very few nodes. This is not solver exhaustion: it is a **proof**. In each z plane (the tile being flat), the corner (0,0) of a box can only be covered by the anchor (0,0) — the other two candidates leave the box — which covers (1,0) and (0,1); then (0,2) can only be covered by the anchor (0,2), which covers (1,2) and (0,3); by induction the whole column 0 and the cells (1, even) are covered by forced anchors, and cell (1,1) has **no** valid candidate left. No 2D rectangle is tileable by the fixed-orientation L, and the flat prism inherits the result plane by plane.

Lesson for family 155: a tile can be **so** constraining that it forbids all tilings — counterexample 155 is the opposite, subtle balance, a tile that tiles Z^3 but forbids only full periodicity.


In [4]:
# B0 -- prisme L x {0} : ne tuile AUCUNE boite alignee (theoreme, cf. prose).
l_prisme: set[Cell] = {(0, 0, 0), (1, 0, 0), (0, 1, 0)}
b0 = [mesurer("L_prisme", l_prisme, w) for w in ((3, 3, 3), (4, 3, 3), (6, 4, 2))]


 L_prisme | fenetre (3, 3, 3) | nodes 4 | NON RESOLU
 L_prisme | fenetre (4, 3, 3) | nodes 4 | NON RESOLU
 L_prisme | fenetre (6, 4, 2) | nodes 5 | NON RESOLU


### 3.3 Experiment B1 — the gapped domino: a proven minimal period, and the window artifact

The gapped domino {(0,0,0),(2,0,0)}: two cells separated by a hole. On each x-line, every cell is either an anchor or an anchor + 2 (exactly once each): writing f(n) ∈ {0,1} for the indicator "n is an anchor", we have **f(n) + f(n−2) = 1 for all n**. The anchor phase is therefore exactly 4-periodic, and **no x-period strictly smaller than 4 exists in any tiling** — a hand-proven constraint, not a measured one. Immediate consequence on the measurement: a window with half-width < 4 can never certify the x-period. We measure (4,2,2) then (8,2,2).

### 3.4 Experiment B2 — the doubled cube: the "looks aperiodic" trap

The doubled cube {0,2}³: the 8 corners of a spaced 2×2×2 cube. Every copy only covers cells of **its parity class** (adding 2 preserves the parity of each coordinate); per class, the tiling reduces (after division by 2) to a tiling by unit cubes. Hence a **unique** tiling of Z^3: anchors {0,1}³ + 4Z³, period group **exactly 4Z³**. On (4,4,4), no period fits in the test box (|vᵢ| ≤ 2): the measured rank drops to **0** — the fully "aperiodic" look — while the true group has rank 3. On (8,8,8), the three generators (4,0,0), (0,4,0), (0,0,4) appear.


In [5]:
# B1 -- domino troue : periode x minimale PROUVEE = 4 (cf. prose 3.3).
domino_troue: set[Cell] = {(0, 0, 0), (2, 0, 0)}
b1_etroite = mesurer("domino_troue", domino_troue, (4, 2, 2))
b1_large = mesurer("domino_troue", domino_troue, (8, 2, 2))

# B2 -- cube dilate : pavage de Z^3 unique, periodes exactement 4Z^3 (cf. prose 3.4).
cube_dilate: set[Cell] = {(x, y, z) for x in (0, 2) for y in (0, 2) for z in (0, 2)}
b2_petite = mesurer("cube_dilate", cube_dilate, (4, 4, 4))
b2_grande = mesurer("cube_dilate", cube_dilate, (8, 8, 8))


 domino_troue | fenetre (4, 2, 2) | nodes 9 | rang 2 | generateurs [(0, 0, 1), (0, 1, 0)]
 domino_troue | fenetre (8, 2, 2) | nodes 17 | rang 3 | generateurs [(0, 0, 1), (0, 1, 0), (4, 0, 0)]
 cube_dilate | fenetre (4, 4, 4) | nodes 9 | rang 0 | generateurs []
 cube_dilate | fenetre (8, 8, 8) | nodes 65 | rang 3 | generateurs [(0, 0, 4), (0, 4, 0), (4, 0, 0)]


### Reading the result

The gapped domino: rank **2** on (4,2,2) — only the partial periods (0,1,0) and (0,0,1) — then rank **3** on (8,2,2) where the generator (4,0,0) becomes visible, exactly the proven minimal period. The doubled cube pushes the same phenomenon to the extreme: rank **0** on (4,4,4) — the most aperiodic look possible — then rank **3** on (8,8,8) with the three generators of 4Z³. Both tiles are **periodic** (their true groups have rank 3); it is the windows that lie, each in its own direction:

- small window ⇒ invisible periods ⇒ measured rank **too small** (B1, B2);
- the converse does not exist: a measured rank 3 is a **certificate** of full periodicity (three periods seen), but a rank < 3 is never a certificate of aperiodicity.


## 4. What a finite window can establish — and what it cannot

A finite window can **never** prove aperiodicity, for a simple reason: any finite tiling observed on W is compatible with a fully periodic tiling of Z^3 (repeat the motif of W). The §3 measurements carry this with numbers:

- the **control** A itself yields rank 2 on a thin window — the half-window ceiling strikes even the simplest tiles;
- the gapped domino (B1) and the doubled cube (B2) have **proven** minimal periods (4 in x for the former, exactly 4Z³ for the latter): their measured ranks 2 and 0 on small windows are **visibility artifacts**, lifted when the window doubles;
- the L prism (B0) recalls the other edge of the spectrum: a local constraint can forbid **every** tiling of a box — there, the solver's failure is a proof, because it is short and structured (forced anchors), not an exhaustion.

What the measurement therefore brings: **it calibrates** (the measured rank is a lower bound capped by the half-window), **it compares** (the backtracking node cost tells whether the local constraint bites), **it traps** (B2 shows that a measured rank 0 says nothing about the true group). What it does not replace: theorem 155, which establishes the absence of rank 3 for **all** tilings of its tile — a universal quantifier that no finite window reaches.

This is the division of labor of the ANALYSE series: the Python notebook **measures and illustrates**, the formalization **proves** — here, the Lean skeleton published by the corpus for family 155.


## 5. Exercises

### Exercise 1 — the period group, not just its rank

The `period_rank` function returns the rank and generators. Write `period_group(placement, window_shape)` returning the **complete list** of period vectors found (in canonical form), then check on the domino that the minimum-L1-norm periods are (1,0,0), (0,1,0) and (0,0,2) — and explain why the step is 2 and not 1 in the long direction.

```python
# TODO student
def period_group(placement, window_shape):
    # Hint: resume period_rank's probing loop and collect
    # ALL invariant nonzero vectors v, instead of stopping at a basis.
    # Step 1: list the candidates in the half-window.
    # Step 2: test invariance (vectorized anchor comparison).
    # Step 3: canonicalize each period and deduplicate.
    return None
```

### Exercise 2 — the smallest window that reveals the x generator

For the gapped domino, the minimal x-period is 4 (proven in §3.3). Find experimentally the smallest window (N₁,2,2) from which the generator (4,0,0) becomes **visible** in the measurement, and check that it is indeed N₁ = 8 (half-window 4). What happens for N₁ = 7?

```python
# TODO student
def plus_petite_fenetre_revelante(tile_cells, axe=0):
    # Hint: sweep increasing N1, call place_tiles then period_rank,
    # stop at the first rank 3. Watch the odd N1 case (half-window
    # floor): does the N1=7 answer surprise you given floor(7/2) = 3?
    return None
```

### Exercise 3 — build a parity constraint

The doubled cube owes its unique tiling to a **parity class** argument. Build another tile whose Z^3 tiling is unique by the same argument (for instance a "doubled L" — take the L's three cells, double every coordinate), prove uniqueness by hand, then check that the measurement yields rank 0 on a small window and rank 3 on a large one, with the expected generators.

```python
# TODO student
def triomino_dilate():
    # Hint: take the L's three cells, multiply every coordinate by 2,
    # and reason by parity class as in 3.4.
    # Step 1: propose the shape. Step 2: prove uniqueness by hand.
    # Step 3: measure on (4,4,4) then (8,8,8) and confront.
    return None
```


## Conclusion

### What has been established

| Result | Measurement |
|---|---|
| The domino is fully periodic | rank 3 on (4,4,4), generators including (0,0,2) |
| A thin window hides even obvious periods | rank 2 on (6,4,2) for that same domino |
| The L prism tiles no aligned box | 0/3 windows solved, short structured failure (theorem) |
| The gapped domino has a proven minimal x-period of 4 | rank 2 on (4,2,2) then 3 on (8,2,2), generator (4,0,0) |
| The doubled cube has a unique tiling with group 4Z³ | rank 0 on (4,4,4) then 3 on (8,8,8) |
| A finite window never proves aperiodicity | §4 argument: repeated motif |

### What remains to the theorem

Statement 155 — a finite tile of Z^3 that tiles with **no** rank-3 tiling — is not re-proven here: it is **presented**, sourced from the openai/math corpus (published Lean skeleton), and illustrated by measured toy cases that each honestly bound their reach. The natural bridge is twofold: the formalization on the Lean side (the corpus's skeleton), and the exact cover on the `sudoku_lean` side — the semantics is the same, only the instances change.

**Leads for the sequel**: the passage to R^3 (unit-cube thickening, arbitrary real translations); the comparison with classical aperiodic tile sets (Berger 1966, Jeandel–Rao in 2D — a *family* of tiles, where 155 gives a *single* tile); and the algorithmic complexity of tiling (undecidability in general, Berger's result).
